# AI Lab Final — All Algorithms

This final notebook contains the complete simple implementations we practiced:

1. BFS — with path
2. DFS — with path
3. Greedy Best First Search (GBFS) — with path
4. A* Search — with path and total cost
5. Hill Climbing
6. Simulated Annealing
7. Gradient Descent — Assignment 5 based
8. Genetic Algorithm — Assignment 7 based

All codes are kept simple and lab-exam friendly.


# 1. BFS — Breadth First Search

Uses a **Queue (FIFO)**.

This version returns the path from `start` to `goal`.


In [ ]:
from collections import deque

def bfs(graph, start, goal):

    queue = deque([(start, [start])])
    visited = set()

    while queue:

        node, path = queue.popleft()

        if node == goal:
            return path

        if node not in visited:

            visited.add(node)

            for neighbor in graph[node]:

                if neighbor not in visited:
                    queue.append((neighbor, path + [neighbor]))

    return None


graph = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F'],
    'D': [],
    'E': [],
    'F': []
}


path = bfs(graph, 'A', 'F')

print("BFS Path:", path)


# 2. DFS — Depth First Search

Uses a **Stack (LIFO)**.

This version also returns the path from `start` to `goal`.


In [ ]:
def dfs(graph, start, goal):

    stack = [(start, [start])]
    visited = set()

    while stack:

        node, path = stack.pop()

        if node == goal:
            return path

        if node not in visited:

            visited.add(node)

            # reversed so left-side node is explored first
            for neighbor in reversed(graph[node]):

                if neighbor not in visited:
                    stack.append((neighbor, path + [neighbor]))

    return None


path = dfs(graph, 'A', 'F')

print("DFS Path:", path)


# 3. Greedy Best First Search (GBFS)

GBFS chooses the node with the **lowest heuristic value**.

Formula:

`f(n) = h(n)`


In [ ]:
import heapq

def gbfs(graph, heuristics, start, goal):

    pq = []

    heapq.heappush(
        pq,
        (heuristics[start], start, [start])
    )

    visited = set()

    while pq:

        h, current, path = heapq.heappop(pq)

        if current in visited:
            continue

        visited.add(current)

        print("Expanding:", current)

        if current == goal:
            return path

        for neighbor in graph[current]:

            if neighbor not in visited:

                heapq.heappush(
                    pq,
                    (
                        heuristics[neighbor],
                        neighbor,
                        path + [neighbor]
                    )
                )

    return None


gbfs_graph = {
    'S': ['A', 'B'],
    'A': ['C', 'D'],
    'B': ['G'],
    'C': [],
    'D': [],
    'G': []
}


heuristics = {
    'S': 7,
    'A': 4,
    'B': 2,
    'C': 5,
    'D': 6,
    'G': 0
}


path = gbfs(
    gbfs_graph,
    heuristics,
    'S',
    'G'
)

print("GBFS Path:", path)


# 4. A* Search

A* uses:

`f(n) = g(n) + h(n)`

Where:

- `g(n)` = actual cost from start
- `h(n)` = estimated cost to goal
- `f(n)` = total estimated cost


In [ ]:
import heapq

def astar(graph, costs, heuristics, start, goal):

    pq = []

    # (f, g, node, path)
    heapq.heappush(
        pq,
        (
            heuristics[start],
            0,
            start,
            [start]
        )
    )

    best_g = {start: 0}

    while pq:

        f, g, current, path = heapq.heappop(pq)

        if g != best_g.get(current):
            continue

        print(
            "Expanding:",
            current,
            "g =", g,
            "h =", heuristics[current],
            "f =", f
        )

        if current == goal:
            return path, g

        for neighbor in graph[current]:

            new_g = g + costs[(current, neighbor)]

            if neighbor not in best_g or new_g < best_g[neighbor]:

                best_g[neighbor] = new_g

                new_f = new_g + heuristics[neighbor]

                heapq.heappush(
                    pq,
                    (
                        new_f,
                        new_g,
                        neighbor,
                        path + [neighbor]
                    )
                )

    return None


astar_graph = {
    'S': ['A', 'B'],
    'A': ['C', 'D'],
    'B': ['D'],
    'C': ['G'],
    'D': ['G'],
    'G': []
}


costs = {
    ('S', 'A'): 2,
    ('S', 'B'): 4,
    ('A', 'C'): 2,
    ('A', 'D'): 5,
    ('B', 'D'): 1,
    ('C', 'G'): 5,
    ('D', 'G'): 3
}


astar_h = {
    'S': 7,
    'A': 6,
    'B': 4,
    'C': 4,
    'D': 2,
    'G': 0
}


result = astar(
    astar_graph,
    costs,
    astar_h,
    'S',
    'G'
)

print("A* Path:", result[0])
print("Total Cost:", result[1])


# 5. Hill Climbing

Steepest-ascent hill climbing:

- Check all immediate neighbors
- Move to the neighbor with the highest value
- Stop when no better neighbor exists


In [ ]:
def hill_climbing(values, neighbors, start):

    current = start
    path = [current]

    while True:

        best = current

        for neighbor in neighbors[current]:

            if values[neighbor] > values[best]:
                best = neighbor

        if best == current:
            return path, current, values[current]

        current = best
        path.append(current)


values = {
    'S': 3,
    'A': 7,
    'B': 5,
    'C': 4,
    'D': 6,
    'E': 9,
    'F': 12,
    'G': 8
}


neighbors = {
    'S': ['A', 'B', 'C'],
    'A': ['S', 'D', 'E'],
    'B': ['S', 'F'],
    'C': ['S', 'G'],
    'D': ['A'],
    'E': ['A'],
    'F': ['B'],
    'G': ['C']
}


path, final_state, final_value = hill_climbing(
    values,
    neighbors,
    'S'
)

print("Path:", path)
print("Stopped at:", final_state)
print("Value:", final_value)


# 6. Simulated Annealing

For minimization:

`delta = current_cost - new_cost`

- If `delta > 0` → accept
- Otherwise accept with probability:

`p = exp(delta / temperature)`


In [ ]:
import math
import random

def cost(x):
    return x ** 2


def simulated_annealing():

    current = random.randint(-10, 10)

    temperature = 100
    cooling_rate = 0.95

    while temperature > 0.1:

        neighbor = current + random.choice([-1, 1])

        current_cost = cost(current)
        new_cost = cost(neighbor)

        delta = current_cost - new_cost

        if delta > 0:
            current = neighbor

        else:

            probability = math.exp(delta / temperature)

            if random.random() < probability:
                current = neighbor

        temperature = temperature * cooling_rate

    return current


result = simulated_annealing()

print("Best Solution:", result)
print("Minimum Cost:", cost(result))


# 7. Gradient Descent — Assignment 5

Minimize:

`J(w) = (w - 4)^2`

Given:

- `w0 = 10`
- `learning rate = 0.25`

Derivative:

`dJ/dw = 2(w - 4)`

Update:

`w_new = w_old - learning_rate * gradient`


In [ ]:
def gradient_descent(w, learning_rate, iterations):

    for i in range(iterations):

        old_w = w

        gradient = 2 * (old_w - 4)

        w = old_w - learning_rate * gradient

        cost = (w - 4) ** 2

        print("Iteration:", i + 1)
        print("Old w:", old_w)
        print("Gradient:", gradient)
        print("New w:", w)
        print("Cost:", cost)
        print("--------------------")

    return w


final_w = gradient_descent(
    w=10,
    learning_rate=0.25,
    iterations=3
)

print("Final w:", final_w)


## Assignment 5 — Large Learning Rate Test

Now use:

`learning_rate = 1.2`

for the first two updates.


In [ ]:
gradient_descent(
    w=10,
    learning_rate=1.2,
    iterations=2
)


# 8. Genetic Algorithm — Assignment 7

3-bit chromosome.

Fitness:

`f(x) = x(7-x)`

Initial population:

`001, 010, 101, 110`

Rules:

1. Calculate fitness
2. Select top two parents
3. One-point crossover after bit 1
4. Flip middle bit of Child 1
5. Calculate offspring fitness
6. Retain best two among parents and offspring


In [ ]:
population = [
    "001",
    "010",
    "101",
    "110"
]


def fitness(chromosome):

    x = int(chromosome, 2)

    return x * (7 - x)


# Step 1: Decode and calculate fitness

print("Initial Population")

for chromosome in population:

    x = int(chromosome, 2)

    print(
        chromosome,
        "x =", x,
        "fitness =", fitness(chromosome)
    )


# Step 2: Selection
# Python sort is stable, so leftmost wins a tie

sorted_population = sorted(
    population,
    key=fitness,
    reverse=True
)

parent1 = sorted_population[0]
parent2 = sorted_population[1]

print("\nSelected Parents:")
print("Parent 1:", parent1)
print("Parent 2:", parent2)


# Step 3: One-point crossover after first bit

child1 = parent1[:1] + parent2[1:]
child2 = parent2[:1] + parent1[1:]

print("\nAfter Crossover:")
print("Child 1:", child1)
print("Child 2:", child2)


# Step 4: Mutation
# Flip the middle bit of Child 1 only

child1_list = list(child1)

if child1_list[1] == '0':
    child1_list[1] = '1'
else:
    child1_list[1] = '0'

child1 = ''.join(child1_list)

print("\nAfter Mutation:")
print("Child 1:", child1)
print("Child 2:", child2)


# Step 5: Offspring fitness

print("\nOffspring Fitness:")
print(child1, "fitness =", fitness(child1))
print(child2, "fitness =", fitness(child2))


# Step 6: Retain best two among parents and offspring

all_candidates = population + [child1, child2]

next_population = sorted(
    all_candidates,
    key=fitness,
    reverse=True
)[:2]

print("\nNext Population:")

for chromosome in next_population:

    print(
        chromosome,
        "x =", int(chromosome, 2),
        "fitness =", fitness(chromosome)
    )


# Quick Memory Table

| Algorithm | Main Idea |
|---|---|
| BFS | Queue, FIFO, level by level |
| DFS | Stack, LIFO, depth first |
| GBFS | Lowest `h(n)` |
| A* | Lowest `g(n)+h(n)` |
| Hill Climbing | Best immediate neighbor |
| Simulated Annealing | Sometimes accepts worse moves |
| Gradient Descent | Move opposite the gradient |
| Genetic Algorithm | Selection → Crossover → Mutation |
